# Unit 5 — Joins & Merges — Data Alignment Surgery 🟡 IMPORTANT
**Phase 2 | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

Real quant work combines:
- **Price data** (daily) + **Earnings** (quarterly) + **Macro indicators** (monthly) + **Alt data** (variable)

**Bad joins = silent bugs**. A duplicated row inflates your sample size. A misaligned date introduces look-ahead bias. A many-to-many join creates phantom data.

> 💡 If you know SQL joins, you already know the logic — this is just pandas syntax.


---
## 1️⃣ The Three Methods — When to Use Each

| Method | Best For | Key Behaviour |
|--------|----------|---------------|
| `.merge()` | Column-based joins (like SQL JOIN) | Joins on shared column values |
| `.join()` | Index-based joins (quick shorthand) | Joins on both DataFrames' index |
| `.concat()` | Stacking rows or columns | Doesn't align — just appends |

### Join Types (same as SQL)
| Type | Result |
|------|--------|
| `inner` | Only rows where key exists in **both** DataFrames ✅ Safest |
| `outer` | All rows from **both** — NaN where no match ⚠️ |
| `left` | All rows from left — right side NaN where no match |
| `right` | All rows from right — left side NaN where no match |


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

# Pull two tickers
spy = yf.download('SPY', start='2022-01-01', auto_adjust=True)['Close'].squeeze()
qqq = yf.download('QQQ', start='2022-01-01', auto_adjust=True)['Close'].squeeze()

# .concat() — stack columns side by side (axis=1)
prices = pd.concat([spy.rename('SPY'), qqq.rename('QQQ')], axis=1)
print(f"Shape: {prices.shape}")
print(prices.tail(3))


In [ ]:
# .join() — index-based (both have DatetimeIndex — cleaner for financial data)
prices_join = spy.rename('SPY').to_frame().join(qqq.rename('QQQ'), how='inner')
print(prices_join.tail(3))


In [ ]:
# .merge() — column-based (better when joining on non-index columns)
spy_df = spy.rename('SPY').reset_index()   # turns index into a 'Date' column
qqq_df = qqq.rename('QQQ').reset_index()

merged = spy_df.merge(qqq_df, on='Date', how='inner')
print(merged.tail(3))


---
## 2️⃣ Validating Your Merge — Always Do This

**Every merge should be followed by validation.** Silent bugs hide here.


In [ ]:
# ✅ Post-merge validation checklist

# 1. Shape check — did rows explode? (sign of many-to-many join)
print(f"SPY rows: {len(spy)}, QQQ rows: {len(qqq)}, Merged rows: {len(prices)}")

# 2. Duplicate index check — the silent killer
dupes = prices.index.duplicated().sum()
print(f"Duplicated rows: {dupes}")  # Should be 0

# 3. NaN check — unexpected missingness
print("\nNaN counts:")
print(prices.isnull().sum())

# 4. Date range check
print(f"\nDate range: {prices.index.min()} to {prices.index.max()}")

# 5. For .merge(), use validate parameter to catch cardinality issues
# 'one_to_one', 'one_to_many', 'many_to_one', 'many_to_many'
# spy_df.merge(qqq_df, on='Date', how='inner', validate='one_to_one')


---
## 3️⃣ Frequency Mismatch — The Hard Problem

**Daily prices + Quarterly earnings = mismatched frequencies.**

You can't just merge naively — you'll get NaN on every non-earnings date.

**Solution: forward-fill (as-of join)**
- Reindex earnings to daily frequency
- Forward-fill until the next report date
- This simulates **what an investor actually knew** at each point in time
- **No look-ahead bias** — future earnings don't appear before their announcement

> ⚠️ There will be NaN rows at the *start* of your series, before the first earnings date. That's correct — you didn't know anything yet.


In [ ]:
# Simulate quarterly earnings (approximate real announcement dates)
earnings_dates = pd.to_datetime([
    '2022-01-28', '2022-04-29', '2022-07-29', '2022-10-28',
    '2023-01-27', '2023-04-28', '2023-07-28', '2023-10-27'
])
eps_values = [1.52, 1.20, 1.29, 1.88, 1.46, 1.26, 1.34, 1.92]

eps = pd.Series(eps_values, index=earnings_dates, name='EPS')
print("Quarterly EPS:")
print(eps)


In [ ]:
# Reindex to daily frequency + forward-fill
# This is the 'as-of' join pattern
daily_eps = eps.reindex(prices.index, method='ffill')

# Merge onto price data
combined = prices.copy()
combined['EPS'] = daily_eps

print("Combined (first 40 rows to see the NaN → fill behaviour):")
print(combined.head(40))
# Notice: EPS is NaN before first earnings date (correct!)
# Then fills forward until next earnings date


---
## 4️⃣ Edge Cases Worth Knowing

| Scenario | Problem | Fix |
|----------|---------|-----|
| Delisted ticker | Missing rows after delisting date | Use `outer` join, then filter by date |
| Duplicate dates | Many-to-many explosion | Use `validate='one_to_one'` in `.merge()` |
| Different timezones | Misaligned index | Normalise timezone before joining |
| Quarterly → Daily | NaN on non-report dates | Reindex + `.ffill()` |


---
## ✅ Self-Check Questions

1. What's the difference between `how='inner'` and `how='outer'`?
   > *Inner: only rows in both. Outer: all rows from both — NaN where no match.*

2. How do you detect if a merge created unintended duplicates?
   > *Check `index.duplicated().sum()` or compare shape before vs after.*

3. When should you use `.join()` vs `.merge()`?
   > *`.join()` when both DataFrames have aligned DatetimeIndex. `.merge()` when joining on column values.*

4. How do you handle quarterly earnings joined to daily prices?
   > *Reindex earnings to daily frequency, then `.ffill()` — simulates as-of knowledge with no look-ahead bias.*

---
## 🧪 Optional Tasks

- Pull SPY and AAPL. Concat side by side. Validate: shape, duplicates, NaN counts.
- Simulate quarterly earnings for any ticker. Forward-fill to daily. Check: are there NaN rows at the start? Why is that correct?
- Try `inner` vs `outer` join between two tickers with different start dates. Count NaN rows. Understand the trade-off.
- Use `validate='one_to_one'` in a `.merge()`. Deliberately create a duplicate date and watch it raise an error.
